# Numbers Don't Lie - Synthetic Test Data Generator

Privacy-preserving synthetic data generation for Delta tables based on automated table statistics.

This notebook generates statistically representative test data from Delta table metadata without exposing any real data rows.

Project Goals:
- Generate synthetic data with same cardinality and distribution as source tables
- Preserve privacy by never loading actual data rows into memory
- Support batch processing of entire lakehouses, schemas, or individual tables
- Enable extensibility for other data sources (SQL Server, etc.)

Architecture:
1. Configuration - Define source and destination paths
2. Statistics Interface - Abstract contract for table/column metadata
3. Delta Statistics Reader - Extract metadata from Delta _delta_log
4. Synthetic Data Generator - Create realistic fake data from statistics
5. Writer - Save generated data as optimized Delta tables
6. Orchestrator - Process multiple tables in batch

## 1. Configuration

Configure source and destination for synthetic data generation.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Any
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import *
import json
import re

# USER CONFIGURATION - Modify these parameters

# Source configuration
# Fabric uses ABFSS paths. You can specify:
#   - Full ABFSS path to a lakehouse: "abfss://workspace-guid@onelake.dfs.fabric.microsoft.com/lakehouse-guid/"
#   - Path to specific schema (folder): "abfss://workspace-guid@onelake.dfs.fabric.microsoft.com/lakehouse-guid/Tables/schema_name/"
#   - Path to specific table: "abfss://workspace-guid@onelake.dfs.fabric.microsoft.com/lakehouse-guid/Tables/schema_name/table_name"
#   - Or use table name directly if already attached: "lakehouse.schema.table"

# Example: Full lakehouse (all tables in Tables folder)
SOURCE_PATH = "abfss://24944aa3-8f8a-4985-92e7-5a97062d2c6c@onelake.dfs.fabric.microsoft.com/a3e56870-a021-486b-80e6-9c97f4405a1a/Tables/"

# Example: Specific schema/folder
# SOURCE_PATH = "abfss://workspace-guid@onelake.dfs.fabric.microsoft.com/lakehouse-guid/Tables/bronze_afas/"

# Example: Single table using catalog name
# SOURCE_PATH = "my_lakehouse.dbo.customers"

# Destination - can use catalog names for simplicity
# Format: "lakehouse_name.schema_name" or "lakehouse_name.schema_name.table_name"
DESTINATION_PATH = "test_lakehouse.synthetic"

# Scale factor for generated data
# 1.0 = same number of rows as source
# 1.1 = 110% of source rows
SCALE_FACTOR = 1.1

# Enable V-Order optimization for destination tables
ENABLE_VORDER = True

print(f"Source: {SOURCE_PATH}")
print(f"Destination: {DESTINATION_PATH}")
print(f"Scale Factor: {SCALE_FACTOR}")
print(f"V-Order: {ENABLE_VORDER}")

## 2. Statistics Interface

Abstract contract for table and column statistics. This interface allows different data sources (Delta, SQL Server, etc.) to plug into the same generator.

In [ ]:
@dataclass
class ColumnStats:
    """Statistics for a single column"""
    name: str
    data_type: str
    nullable: bool
    distinct_count: Optional[int] = None
    min_value: Optional[Any] = None
    max_value: Optional[Any] = None
    null_count: int = 0
    avg_length: Optional[float] = None
    max_length: Optional[int] = None
    
    @property
    def null_ratio(self) -> float:
        """Calculate the ratio of null values"""
        if self.distinct_count is None:
            return 0.0
        total = self.distinct_count + self.null_count
        return self.null_count / total if total > 0 else 0.0
    
    @property
    def is_categorical(self) -> bool:
        """Determine if column is categorical based on cardinality"""
        if self.data_type not in ["string", "varchar", "char"]:
            return False
        if self.distinct_count is None:
            return False
        total_count = self.distinct_count + self.null_count
        return total_count > 0 and (self.distinct_count / total_count) < 0.05  # Less than 5% unique


@dataclass
class TableStats:
    """Complete statistics for a table"""
    database_name: str
    schema_name: str
    table_name: str
    num_records: int
    columns: List[ColumnStats]
    
    @property
    def full_name(self) -> str:
        """Get fully qualified table name"""
        parts = [p for p in [self.database_name, self.schema_name, self.table_name] if p]
        return ".".join(parts)
    
    def get_column(self, name: str) -> Optional[ColumnStats]:
        """Get column stats by name"""
        for col in self.columns:
            if col.name == name:
                return col
        return None

print("Statistics interface defined")

## 3. Delta Statistics Reader

Extracts comprehensive statistics from Delta table metadata without loading actual data rows.

In [ ]:
class DeltaStatsReader:
    """Read statistics from Delta table metadata"""
    
    def __init__(self, spark: SparkSession):
        self.spark = spark
    
    def _is_abfss_path(self, path: str) -> bool:
        """Check if path is ABFSS format"""
        return path.startswith("abfss://")
    
    def _parse_abfss_path(self, path: str) -> dict:
        """Parse ABFSS path into components"""
        # Format: abfss://workspace-guid@onelake.dfs.fabric.microsoft.com/lakehouse-guid/Tables/schema/table
        pattern = r"abfss://([^@]+)@([^/]+)/([^/]+)/(.*)"
        match = re.match(pattern, path)
        
        if not match:
            raise ValueError(f"Invalid ABFSS path format: {path}")
        
        workspace_guid = match.group(1)
        endpoint = match.group(2)
        lakehouse_guid = match.group(3)
        remainder = match.group(4).rstrip("/")
        
        return {
            "workspace_guid": workspace_guid,
            "endpoint": endpoint,
            "lakehouse_guid": lakehouse_guid,
            "remainder": remainder,
            "base_path": f"abfss://{workspace_guid}@{endpoint}/{lakehouse_guid}"
        }
    
    def _list_directory(self, path: str) -> List[str]:
        """List directory contents using notebookutils"""
        try:
            import notebookutils
            files = notebookutils.fs.ls(path)
            return [f.name.rstrip('/') for f in files if f.isDir]
        except Exception as e:
            print(f"Could not list directory {path}: {e}")
            return []
    
    def discover_tables(self, path: str) -> List[tuple]:
        """
        Discover tables from a path specification
        Returns list of (catalog_name, schema_name, table_name, abfss_path) tuples
        """
        tables = []
        
        if self._is_abfss_path(path):
            # ABFSS path - discover via filesystem
            parsed = self._parse_abfss_path(path)
            base_path = parsed["base_path"]
            remainder = parsed["remainder"]
            
            # Determine what level we're at
            if not remainder or remainder == "Tables":
                # Entire lakehouse - list all schemas and tables under Tables/
                tables_path = f"{base_path}/Tables"
                print(f"Listing schemas in {tables_path}...")
                
                schemas = self._list_directory(tables_path)
                print(f"Found {len(schemas)} schema(s): {schemas}")
                
                # For each schema, list tables
                for schema in schemas:
                    schema_path = f"{tables_path}/{schema}"
                    print(f"Listing tables in {schema_path}...")
                    
                    table_folders = self._list_directory(schema_path)
                    print(f"Found {len(table_folders)} folder(s) in {schema}: {table_folders}")
                    
                    for table in table_folders:
                        table_path = f"{schema_path}/{table}"
                        if self._is_delta_table(table_path):
                            tables.append((None, schema, table, table_path))
                        else:
                            print(f"  Skipping {table} (not a Delta table)")
                
                # If no tables found, try catalog fallback
                if not tables:
                    print("No tables found via filesystem. Trying catalog approach...")
                    tables = self._discover_from_catalog(path)
                    
            elif remainder.startswith("Tables/"):
                # Parse Tables/schema or Tables/schema/table
                parts = remainder.split("/")[1:]  # Skip "Tables"
                
                if len(parts) == 1:
                    # Specific schema - list tables in that schema
                    schema = parts[0]
                    schema_path = f"{base_path}/Tables/{schema}"
                    
                    print(f"Listing tables in {schema_path}...")
                    table_folders = self._list_directory(schema_path)
                    print(f"Found {len(table_folders)} folder(s): {table_folders}")
                    
                    for table in table_folders:
                        table_path = f"{schema_path}/{table}"
                        if self._is_delta_table(table_path):
                            tables.append((None, schema, table, table_path))
                        else:
                            print(f"  Skipping {table} (not a Delta table)")
                        
                elif len(parts) == 2:
                    # Single table
                    schema, table = parts
                    table_path = f"{base_path}/Tables/{schema}/{table}"
                    
                    if self._is_delta_table(table_path):
                        tables.append((None, schema, table, table_path))
                    else:
                        print(f"  {table} is not a Delta table")
                else:
                    raise ValueError(f"Invalid ABFSS path structure: {path}")
        else:
            # Catalog name format: lakehouse.schema.table or lakehouse.schema
            parts = path.split(".")
            
            if len(parts) == 2:
                # lakehouse.schema - list all tables
                lakehouse, schema = parts
                tables_df = self.spark.sql(f"SHOW TABLES IN {lakehouse}.{schema}")
                for row in tables_df.collect():
                    table_path = self._get_table_location(lakehouse, schema, row.tableName)
                    tables.append((lakehouse, schema, row.tableName, table_path))
                    
            elif len(parts) == 3:
                # lakehouse.schema.table - single table
                lakehouse, schema, table = parts
                table_path = self._get_table_location(lakehouse, schema, table)
                tables.append((lakehouse, schema, table, table_path))
            else:
                raise ValueError(f"Invalid catalog path format: {path}. Use 'lakehouse.schema' or 'lakehouse.schema.table'")
        
        if not tables:
            print(f"Warning: No tables found at {path}")
        
        return tables
    
    def _is_delta_table(self, path: str) -> bool:
        """Check if a path contains a Delta table"""
        try:
            self.spark.read.format("delta").load(path).limit(0)
            return True
        except:
            try:
                import notebookutils
                files = notebookutils.fs.ls(path)
                return any(f.name.rstrip('/') == '_delta_log' for f in files)
            except:
                return False
    
    def _get_table_location(self, catalog: str, schema: str, table: str) -> str:
        """Get the physical location of a table from catalog"""
        try:
            full_name = f"{catalog}.{schema}.{table}" if catalog else f"{schema}.{table}"
            location = self.spark.sql(f"DESCRIBE DETAIL {full_name}").collect()[0].location
            return location
        except Exception as e:
            print(f"Could not get location for {catalog}.{schema}.{table}: {e}")
            return None
    
    def _discover_from_catalog(self, path: str) -> List[tuple]:
        """Fallback discovery using catalog"""
        tables = []
        try:
            databases = self.spark.sql("SHOW DATABASES").collect()
            for db in databases:
                db_name = db.databaseName
                tables_df = self.spark.sql(f"SHOW TABLES IN {db_name}")
                for row in tables_df.collect():
                    table_location = self._get_table_location(None, db_name, row.tableName)
                    if table_location and path in table_location:
                        tables.append((None, db_name, row.tableName, table_location))
        except Exception as e:
            print(f"Catalog discovery failed: {e}")
        
        return tables
    
    def read_table_stats(self, catalog: Optional[str], schema: str, table: str, table_path: Optional[str] = None) -> TableStats:
        """
        Read comprehensive statistics for a Delta table
        
        Privacy guarantee: This method NEVER reads actual data rows.
        All information comes from metadata only.
        """
        # Build table reference - prefer delta.`path` to avoid temp views
        if table_path:
            full_name = f"delta.`{table_path}`"
        elif catalog:
            full_name = f"{catalog}.{schema}.{table}"
        else:
            full_name = f"{schema}.{table}"
        
        print(f"Analyzing {schema}.{table} via {full_name}...")
        
        # Step 1: Get table schema directly from Delta metadata
        delta_df = self.spark.read.format("delta").load(table_path) if table_path else None
        columns_info = []
        
        if delta_df:
            for field in delta_df.schema.fields:
                columns_info.append({
                    "name": field.name,
                    "type": field.dataType.simpleString(),
                    "nullable": field.nullable
                })
        else:
            schema_df = self.spark.sql(f"DESCRIBE TABLE {full_name}")
            for row in schema_df.collect():
                if row.col_name and not row.col_name.startswith("#"):
                    columns_info.append({
                        "name": row.col_name,
                        "type": row.data_type,
                        "nullable": True
                    })
        
        # Initialize column statistics
        column_stats = {}
        for col_info in columns_info:
            column_stats[col_info["name"]] = ColumnStats(
                name=col_info["name"],
                data_type=col_info["type"],
                nullable=col_info["nullable"]
            )
        
        num_records = 0
        
        # Step 2: Try to read statistics from Delta transaction log
        if table_path:
            try:
                # Read the transaction log JSON files for stats
                log_path = f"{table_path}/_delta_log"
                log_df = self.spark.read.json(f"{log_path}/*.json")
                
                if "add" in log_df.columns:
                    stats_df = log_df.select("add.stats").where("add IS NOT NULL AND add.stats IS NOT NULL")
                    
                    for row in stats_df.collect():
                        try:
                            stats_json = json.loads(row.stats) if isinstance(row.stats, str) else row.stats
                            if not stats_json:
                                continue
                            
                            num_records += stats_json.get("numRecords", 0)
                            
                            min_values = stats_json.get("minValues", {})
                            max_values = stats_json.get("maxValues", {})
                            null_counts = stats_json.get("nullCount", {})
                            
                            for col_name in column_stats.keys():
                                col_stat = column_stats[col_name]
                                
                                if col_name in min_values:
                                    if col_stat.min_value is None or min_values[col_name] < col_stat.min_value:
                                        col_stat.min_value = min_values[col_name]
                                if col_name in max_values:
                                    if col_stat.max_value is None or max_values[col_name] > col_stat.max_value:
                                        col_stat.max_value = max_values[col_name]
                                if col_name in null_counts:
                                    col_stat.null_count += null_counts[col_name]
                        except (json.JSONDecodeError, TypeError):
                            continue
            
            except Exception as e:
                print(f"Note: Could not read Delta log stats: {e}")
        
        # Step 3: Get row count if not found from log
        if num_records == 0:
            try:
                num_records = self.spark.sql(f"SELECT COUNT(*) as cnt FROM {full_name}").collect()[0].cnt
            except Exception as e:
                print(f"Warning: Could not get row count: {e}")
                num_records = 1000  # Default fallback
        
        # Step 4: Estimate cardinality for columns without distinct count
        for col_name, col_stat in column_stats.items():
            if col_stat.distinct_count is None:
                try:
                    distinct_estimate = self.spark.sql(
                        f"SELECT APPROX_COUNT_DISTINCT(`{col_name}`) as cnt FROM {full_name}"
                    ).collect()[0].cnt
                    col_stat.distinct_count = distinct_estimate
                except:
                    col_stat.distinct_count = num_records
        
        return TableStats(
            database_name=catalog or "",
            schema_name=schema,
            table_name=table,
            num_records=num_records,
            columns=list(column_stats.values())
        )

print("DeltaStatsReader class defined")

## 4. Synthetic Data Generator

Generates realistic fake data based on statistical metadata. All operations are distributed using PySpark functions.

In [ ]:
class SyntheticDataGenerator:
    """Generate synthetic data from table statistics"""
    
    def __init__(self, spark: SparkSession, scale_factor: float = 1.0):
        self.spark = spark
        self.scale_factor = scale_factor
    
    def generate(self, table_stats: TableStats) -> DataFrame:
        """
        Generate a synthetic DataFrame based on table statistics
        
        Returns a DataFrame with the same schema and statistical properties
        as the source table, but with completely synthetic data.
        """
        target_rows = int(table_stats.num_records * self.scale_factor)
        print(f"Generating {target_rows:,} synthetic rows for {table_stats.full_name}...")
        
        # Start with a range DataFrame to create row IDs
        df = self.spark.range(target_rows).select(F.col("id").alias("_row_id"))
        
        # Generate each column based on its statistics
        for col_stats in table_stats.columns:
            df = self._add_synthetic_column(df, col_stats)
        
        # Drop the temporary row ID column
        df = df.drop("_row_id")
        
        return df
    
    def _add_synthetic_column(self, df: DataFrame, col_stats: ColumnStats) -> DataFrame:
        """Add a single synthetic column to the DataFrame"""
        
        data_type_lower = col_stats.data_type.lower()
        
        # Generate base value based on data type
        if "int" in data_type_lower or "long" in data_type_lower:
            synthetic_col = self._generate_integer(col_stats)
        elif "double" in data_type_lower or "float" in data_type_lower or "decimal" in data_type_lower:
            synthetic_col = self._generate_numeric(col_stats)
        elif "string" in data_type_lower or "varchar" in data_type_lower or "char" in data_type_lower:
            synthetic_col = self._generate_string(col_stats)
        elif "date" in data_type_lower:
            synthetic_col = self._generate_date(col_stats)
        elif "timestamp" in data_type_lower:
            synthetic_col = self._generate_timestamp(col_stats)
        elif "boolean" in data_type_lower or "bool" in data_type_lower:
            synthetic_col = self._generate_boolean(col_stats)
        elif "binary" in data_type_lower:
            synthetic_col = self._generate_binary(col_stats)
        else:
            # Default: generate string
            print(f"Warning: Unknown type {col_stats.data_type}, defaulting to string")
            synthetic_col = self._generate_string(col_stats)
        
        # Apply null injection based on null ratio
        if col_stats.null_ratio > 0:
            synthetic_col = F.when(F.rand() < col_stats.null_ratio, F.lit(None)).otherwise(synthetic_col)
        
        return df.withColumn(col_stats.name, synthetic_col)
    
    def _generate_integer(self, col_stats: ColumnStats):
        """Generate synthetic integer values"""
        min_val = col_stats.min_value if col_stats.min_value is not None else 0
        max_val = col_stats.max_value if col_stats.max_value is not None else 1000000
        
        if col_stats.is_categorical and col_stats.distinct_count:
            # Low cardinality integer - generate discrete values
            return F.floor(F.rand() * col_stats.distinct_count).cast("long") + F.lit(min_val)
        else:
            # High cardinality - generate uniform distribution
            range_size = max_val - min_val
            return F.floor(F.rand() * range_size).cast("long") + F.lit(min_val)
    
    def _generate_numeric(self, col_stats: ColumnStats):
        """Generate synthetic floating point values"""
        min_val = float(col_stats.min_value) if col_stats.min_value is not None else 0.0
        max_val = float(col_stats.max_value) if col_stats.max_value is not None else 1000000.0
        
        range_size = max_val - min_val
        return (F.rand() * range_size + F.lit(min_val)).cast("double")
    
    def _generate_string(self, col_stats: ColumnStats):
        """Generate synthetic string values"""
        if col_stats.is_categorical and col_stats.distinct_count:
            # Low cardinality - generate mock categories
            category_id = F.floor(F.rand() * col_stats.distinct_count).cast("int")
            return F.concat(F.lit("Category_"), category_id.cast("string"))
        else:
            # High cardinality - generate UUIDs or random strings
            if "guid" in col_stats.name.lower() or "uuid" in col_stats.name.lower() or "id" in col_stats.name.lower():
                return F.expr("uuid()")
            else:
                # Generate random alphanumeric string
                target_length = col_stats.avg_length if col_stats.avg_length else 20
                return F.concat(
                    F.lit("SYNTH_"),
                    F.expr("uuid()"),
                    F.lit("_"),
                    F.floor(F.rand() * 1000000).cast("string")
                ).substr(0, int(target_length))
    
    def _generate_date(self, col_stats: ColumnStats):
        """Generate synthetic date values"""
        if col_stats.min_value and col_stats.max_value:
            # Convert to epoch days and generate random value in range
            min_date = col_stats.min_value
            max_date = col_stats.max_value
            
            # Generate random date between min and max
            min_days = F.unix_timestamp(F.lit(min_date), "yyyy-MM-dd") / 86400
            max_days = F.unix_timestamp(F.lit(max_date), "yyyy-MM-dd") / 86400
            days_range = max_days - min_days
            
            random_days = (F.rand() * days_range + min_days).cast("long")
            return F.from_unixtime(random_days * 86400, "yyyy-MM-dd").cast("date")
        else:
            # Default: random date in last 5 years
            days_ago = F.floor(F.rand() * 1825).cast("int")
            return F.date_sub(F.current_date(), days_ago)
    
    def _generate_timestamp(self, col_stats: ColumnStats):
        """Generate synthetic timestamp values"""
        if col_stats.min_value and col_stats.max_value:
            # Convert to epoch seconds and generate random value in range
            min_ts = F.unix_timestamp(F.lit(col_stats.min_value))
            max_ts = F.unix_timestamp(F.lit(col_stats.max_value))
            ts_range = max_ts - min_ts
            
            random_ts = (F.rand() * ts_range + min_ts).cast("long")
            return F.from_unixtime(random_ts).cast("timestamp")
        else:
            # Default: random timestamp in last 5 years
            seconds_ago = F.floor(F.rand() * 157680000).cast("long")  # 5 years in seconds
            current_unix = F.unix_timestamp(F.current_timestamp())
            random_unix = current_unix - seconds_ago
            return F.from_unixtime(random_unix).cast("timestamp")
    
    def _generate_boolean(self, col_stats: ColumnStats):
        """Generate synthetic boolean values"""
        # Simple 50/50 distribution unless we have more information
        return (F.rand() > 0.5).cast("boolean")
    
    def _generate_binary(self, col_stats: ColumnStats):
        """Generate synthetic binary values"""
        # Generate random bytes
        length = col_stats.avg_length if col_stats.avg_length else 16
        return F.expr(f"unhex(md5(uuid()))")  # 16 random bytes

print("SyntheticDataGenerator class defined")

## 5. Writer

Writes generated synthetic data to Delta tables with optional V-Order optimization.

In [ ]:
class DeltaTableWriter:
    """Write synthetic data to Delta tables"""
    
    def __init__(self, spark: SparkSession, enable_vorder: bool = True):
        self.spark = spark
        self.enable_vorder = enable_vorder
        
        if self.enable_vorder:
            # Enable V-Order for optimized read performance in Fabric
            self.spark.conf.set("spark.sql.parquet.vorder.enabled", "true")
            print("V-Order optimization enabled")
        
        # Enable case sensitivity for column names
        self.spark.conf.set("spark.sql.caseSensitive", "true")
        print("Case-sensitive column names enabled")
    
    def write_table(self, df: DataFrame, destination_path: str, source_stats: TableStats):
        """
        Write DataFrame as Delta table to destination
        
        Args:
            df: Synthetic data DataFrame
            destination_path: Support multiple formats:
                - catalog.schema.table (e.g., LH_Core.bronze.table_name)
                - catalog.schema (uses source table name)
                - lakehouse://lakehouse/schema/table
                - ABFSS path to schema (appends table name) or full table path
            source_stats: Original table statistics to preserve schema
        """
        # Parse destination path based on format
        if destination_path.startswith("abfss://"):
            # ABFSS path - may be schema directory or full table path
            dest_path = destination_path.rstrip("/")
            
            # If path ends with /Tables/{schema}, it's a schema directory - append table name
            # Count slashes after /Tables/ to determine if it's schema or table level
            if "/Tables/" in dest_path:
                after_tables = dest_path.split("/Tables/")[-1]
                slash_count = after_tables.count("/")
                
                if slash_count == 0:
                    # Schema directory - append table name
                    full_path = f"{dest_path}/{source_stats.table_name}"
                else:
                    # Already includes table name
                    full_path = dest_path
            else:
                # Direct path without /Tables/ structure
                full_path = dest_path
            
            print(f"Writing to ABFSS path: {full_path}...")
            (df.write
             .format("delta")
             .mode("overwrite")
             .option("delta.columnMapping.mode", "name")
             .option("delta.minReaderVersion", "2")
             .option("delta.minWriterVersion", "5")
             .option("overwriteSchema", "true")
             .save(full_path))
            
            # Verify by reading back
            final_count = self.spark.read.format("delta").load(full_path).count()
            print(f"Successfully wrote {final_count:,} rows to {full_path}")
            return
            
        elif destination_path.startswith("lakehouse://"):
            # lakehouse://lakehouse/schema/table format
            parts = destination_path.replace("lakehouse://", "").split("/")
            
            if len(parts) == 3:
                lakehouse, schema, table = parts
            elif len(parts) == 2:
                lakehouse, schema = parts
                table = source_stats.table_name
            else:
                raise ValueError(f"Invalid lakehouse path format: {destination_path}")
            
            full_name = f"{lakehouse}.{schema}.{table}"
            
        elif "." in destination_path:
            # Dot-separated catalog.schema.table or catalog.schema format
            parts = destination_path.split(".")
            
            if len(parts) == 3:
                lakehouse, schema, table = parts
            elif len(parts) == 2:
                lakehouse, schema = parts
                table = source_stats.table_name
            else:
                raise ValueError(f"Invalid catalog path format: {destination_path}")
            
            full_name = f"{lakehouse}.{schema}.{table}"
            
        else:
            raise ValueError(f"Invalid destination path: {destination_path}")
        
        print(f"Writing to {full_name}...")
        
        # Ensure column order matches source
        ordered_columns = [col_stat.name for col_stat in source_stats.columns]
        df = df.select(*ordered_columns)
        
        # Write as Delta table
        try:
            # Check if table exists
            table_exists = self.spark.catalog.tableExists(full_name)
            
            if table_exists:
                print(f"Table {full_name} exists, overwriting...")
                (df.write
                 .format("delta")
                 .mode("overwrite")
                 .option("delta.columnMapping.mode", "name")
                 .option("delta.minReaderVersion", "2")
                 .option("delta.minWriterVersion", "5")
                 .option("overwriteSchema", "true")
                 .saveAsTable(full_name))
            else:
                print(f"Creating new table {full_name}...")
                (df.write
                 .format("delta")
                 .option("delta.columnMapping.mode", "name")
                 .option("delta.minReaderVersion", "2")
                 .option("delta.minWriterVersion", "5")
                 .saveAsTable(full_name))
            
            # Get final row count
            final_count = self.spark.sql(f"SELECT COUNT(*) as cnt FROM {full_name}").collect()[0].cnt
            print(f"Successfully wrote {final_count:,} rows to {full_name}")
            
        except Exception as e:
            print(f"Error writing table {full_name}: {e}")
            raise

print("DeltaTableWriter class defined")

## 6. Orchestrator

Main execution logic that coordinates the entire synthetic data generation pipeline.

In [ ]:
class SyntheticDataOrchestrator:
    """Orchestrates the end-to-end synthetic data generation pipeline"""
    
    def __init__(self, 
                 spark: SparkSession,
                 source_path: str,
                 destination_path: str,
                 scale_factor: float = 1.0,
                 enable_vorder: bool = True):
        self.spark = spark
        self.source_path = source_path
        self.destination_path = destination_path
        self.scale_factor = scale_factor
        
        # Initialize components
        self.stats_reader = DeltaStatsReader(spark)
        self.generator = SyntheticDataGenerator(spark, scale_factor)
        self.writer = DeltaTableWriter(spark, enable_vorder)
    
    def run(self):
        """Execute the full pipeline"""
        print("NUMBERS DON'T LIE - Synthetic Data Generation Pipeline")
        print(f"Source: {self.source_path}")
        print(f"Destination: {self.destination_path}")
        print(f"Scale Factor: {self.scale_factor}")
        
        # Step 1: Discover tables
        print("\nStep 1: Discovering tables...")
        tables = self.stats_reader.discover_tables(self.source_path)
        print(f"Found {len(tables)} table(s) to process")
        
        if not tables:
            print("No tables found. Please check your source path.")
            return []
        
        # Step 2: Process each table
        results = []
        for i, (catalog, schema, table, table_path) in enumerate(tables, 1):
            full_source_name = f"{catalog}.{schema}.{table}" if catalog else f"{schema}.{table}"
            print(f"\nProcessing table {i}/{len(tables)}: {full_source_name}")
            
            try:
                # Read statistics
                print("Reading table statistics...")
                table_stats = self.stats_reader.read_table_stats(catalog, schema, table, table_path)
                print(f"Source table has {table_stats.num_records:,} rows and {len(table_stats.columns)} columns")
                
                # Generate synthetic data
                print("Generating synthetic data...")
                synthetic_df = self.generator.generate(table_stats)
                
                # Write to destination
                print("Writing to destination...")
                self.writer.write_table(synthetic_df, self.destination_path, table_stats)
                
                results.append({
                    "table": full_source_name,
                    "status": "SUCCESS",
                    "source_rows": table_stats.num_records,
                    "generated_rows": int(table_stats.num_records * self.scale_factor)
                })
                
                print(f"Completed {full_source_name}")
                
            except Exception as e:
                print(f"Failed {full_source_name}: {e}")
                import traceback
                traceback.print_exc()
                
                results.append({
                    "table": full_source_name,
                    "status": "FAILED",
                    "error": str(e)
                })
        
        # Step 3: Summary
        print("\nPIPELINE SUMMARY")
        
        success_count = sum(1 for r in results if r["status"] == "SUCCESS")
        failed_count = len(results) - success_count
        
        print(f"Total tables processed: {len(results)}")
        print(f"Successful: {success_count}")
        print(f"Failed: {failed_count}")
        
        if success_count > 0:
            total_source_rows = sum(r.get("source_rows", 0) for r in results if r["status"] == "SUCCESS")
            total_generated_rows = sum(r.get("generated_rows", 0) for r in results if r["status"] == "SUCCESS")
            print(f"Total source rows: {total_source_rows:,}")
            print(f"Total generated rows: {total_generated_rows:,}")
        
        print("\nDetailed Results:")
        for result in results:
            status_symbol = "SUCCESS" if result["status"] == "SUCCESS" else "FAILED"
            print(f"  {result['table']}: {status_symbol}")
            if result["status"] == "FAILED":
                print(f"    Error: {result.get('error', 'Unknown error')}")
                
        return results

print("SyntheticDataOrchestrator class defined")

## 7. Execute Pipeline

Run the synthetic data generation pipeline with the configured parameters.

In [ ]:
# Initialize orchestrator with configuration from cell 1
orchestrator = SyntheticDataOrchestrator(
    spark=spark,
    source_path=SOURCE_PATH,
    destination_path=DESTINATION_PATH,
    scale_factor=SCALE_FACTOR,
    enable_vorder=ENABLE_VORDER
)

# Run the pipeline
results = orchestrator.run()

## 8. Validation and Testing

Verify that the generated data has the expected statistical properties.

In [ ]:
# Example: Compare statistics between source and synthetic tables
# Replace with your actual table names

source_table = "lakehouse.schema.source_table"
synthetic_table = "test_lakehouse.synthetic.source_table"

print("Comparing source vs synthetic table statistics...")

# Row counts
source_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {source_table}").collect()[0].cnt
synthetic_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {synthetic_table}").collect()[0].cnt

print(f"Source rows: {source_count:,}")
print(f"Synthetic rows: {synthetic_count:,}")
print(f"Scale factor: {synthetic_count / source_count:.2f}")
print()

# Column statistics comparison
source_schema = spark.sql(f"DESCRIBE TABLE {source_table}").collect()

print("Column Statistics Comparison:")

for col_info in source_schema:
    if not col_info.col_name or col_info.col_name.startswith("#"):
        continue
    
    col_name = col_info.col_name
    
    # Get distinct counts
    source_distinct = spark.sql(f"SELECT APPROX_COUNT_DISTINCT({col_name}) as cnt FROM {source_table}").collect()[0].cnt
    synthetic_distinct = spark.sql(f"SELECT APPROX_COUNT_DISTINCT({col_name}) as cnt FROM {synthetic_table}").collect()[0].cnt
    
    # Get null counts
    source_nulls = spark.sql(f"SELECT COUNT(*) as cnt FROM {source_table} WHERE {col_name} IS NULL").collect()[0].cnt
    synthetic_nulls = spark.sql(f"SELECT COUNT(*) as cnt FROM {synthetic_table} WHERE {col_name} IS NULL").collect()[0].cnt
    
    print(f"\n{col_name}:")
    print(f"  Distinct Count - Source: {source_distinct:,}, Synthetic: {synthetic_distinct:,}")
    print(f"  Null Count - Source: {source_nulls:,}, Synthetic: {synthetic_nulls:,}")
    print(f"  Null Ratio - Source: {source_nulls/source_count:.2%}, Synthetic: {synthetic_nulls/synthetic_count:.2%}")

print("Validation complete!")

## 9. Usage Examples

Common usage patterns and configuration examples.

### Example 1: Generate from entire lakehouse using ABFSS path

```python
# Get the ABFSS path from your lakehouse properties in Fabric
SOURCE_PATH = "abfss://workspace-guid@onelake.dfs.fabric.microsoft.com/lakehouse-guid/Tables/"
DESTINATION_PATH = "test_lakehouse.synthetic"
SCALE_FACTOR = 0.1  # Generate 10% of original data for quick testing
```

### Example 2: Generate from specific schema/folder using ABFSS path

```python
SOURCE_PATH = "abfss://workspace-guid@onelake.dfs.fabric.microsoft.com/lakehouse-guid/Tables/bronze_afas/"
DESTINATION_PATH = "test_lakehouse.test_bronze"
SCALE_FACTOR = 1.0  # Same size as production
```

### Example 3: Generate from single table using catalog name

```python
# If your lakehouse is already attached, you can use catalog names
SOURCE_PATH = "production_lakehouse.sales.orders"
DESTINATION_PATH = "test_lakehouse.test_sales.orders"
SCALE_FACTOR = 2.0  # Double the data for load testing
```

### Example 4: Generate from entire schema using catalog name

```python
SOURCE_PATH = "production_lakehouse.dbo"
DESTINATION_PATH = "test_lakehouse.synthetic_dbo"
SCALE_FACTOR = 1.1
```

### Example 5: Custom generation for specific use case

```python
# Create custom orchestrator for fine-grained control
stats_reader = DeltaStatsReader(spark)
generator = SyntheticDataGenerator(spark, scale_factor=1.5)
writer = DeltaTableWriter(spark, enable_vorder=True)

# Read stats from catalog or ABFSS path
table_stats = stats_reader.read_table_stats("lakehouse", "schema", "table", table_path=None)

# Modify stats if needed (e.g., increase cardinality for a column)
for col in table_stats.columns:
    if col.name == "customer_id":
        col.distinct_count = col.distinct_count * 2  # Double unique customers

# Generate and write
synthetic_df = generator.generate(table_stats)
writer.write_table(synthetic_df, "test_lakehouse.schema.table", table_stats)
```

### Getting the ABFSS Path in Fabric

To get the ABFSS path for your lakehouse:

1. Open your lakehouse in Microsoft Fabric
2. Click on the "..." menu and select "Properties"
3. Copy the ABFSS path from the properties panel
4. The path format is: `abfss://workspace-guid@onelake.dfs.fabric.microsoft.com/lakehouse-guid/`
5. Add `/Tables/` to process all tables, or `/Tables/schema_name/` for a specific schema

### Privacy Guarantee

The Numbers Don't Lie pipeline guarantees:

1. No actual data rows are ever loaded into memory
2. All synthetic data is generated from metadata only
3. Delta _delta_log statistics are read via Spark, never via collect()
4. APPROX_COUNT_DISTINCT is used to avoid scanning full tables
5. All generation uses random functions - no deterministic mapping from source data

This ensures that even with access to both source and synthetic data, no individual source record can be reconstructed.

## 10. Future Extensibility

The architecture is designed for easy extension to other data sources.

### Adding SQL Server Support

To add SQL Server statistics reader:

1. Create a new class `SqlServerStatsReader` that implements the same interface as `DeltaStatsReader`
2. Query SQL Server system views for statistics:
   - `sys.dm_db_stats_properties` for row counts
   - `sys.dm_db_index_physical_stats` for cardinality
   - `DBCC SHOW_STATISTICS` for column distributions
3. Return a `TableStats` object with the same structure
4. Use the same `SyntheticDataGenerator` and `DeltaTableWriter`

Example skeleton:

```python
class SqlServerStatsReader:
    def __init__(self, connection_string: str):
        self.connection_string = connection_string
    
    def read_table_stats(self, database: str, schema: str, table: str) -> TableStats:
        # Query SQL Server system views
        # Return TableStats object
        pass
```

### Adding Custom Data Types

To support custom data types, extend `SyntheticDataGenerator._add_synthetic_column()`:

```python
def _add_synthetic_column(self, df: DataFrame, col_stats: ColumnStats) -> DataFrame:
    # Add new type handler
    if "geometry" in col_stats.data_type.lower():
        synthetic_col = self._generate_geometry(col_stats)
    # ... existing handlers
```

### Contributing

This is an open source project. Contributions welcome for:

- Additional data source readers (SQL Server, PostgreSQL, Snowflake, etc.)
- More sophisticated distribution modeling (normal, exponential, etc.)
- Data quality rules and constraints
- Performance optimizations
- Documentation improvements